In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

print("Booting up Data Pipeline...")

# 1. Generate Realistic NER Terrain Data
np.random.seed(42)
n_samples = 1000

# Features mapped to Meghalaya/Assam geography
data = {
    'rainfall_72h_mm': np.random.uniform(10, 350, n_samples), # Heavy monsoon rain
    'slope_angle_deg': np.random.uniform(0, 65, n_samples),   # Steep terrain
    'elevation_m': np.random.uniform(50, 2000, n_samples),    # Sea level to high hills
    'soil_moisture_index': np.random.uniform(0.1, 1.0, n_samples)
}
df = pd.DataFrame(data)

# Risk Equation: Landslides trigger when high rain meets steep slopes and wet soil
risk_score = (df['rainfall_72h_mm'] * 0.4) + (df['slope_angle_deg'] * 2.5) + (df['soil_moisture_index'] * 50)
df['landslide_occurred'] = np.where(risk_score > 180, 1, 0)

# 2. Prepare for Training
X = df[['rainfall_72h_mm', 'slope_angle_deg', 'elevation_m', 'soil_moisture_index']]
y = df['landslide_occurred']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train the XGBoost Engine
print("Training XGBoost Classifier...")
model = xgb.XGBClassifier(eval_metric='logloss')
model.fit(X_train, y_train)

# 4. Evaluate & Save
predictions = model.predict(X_test)
print(f"✅ Model Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%\n")

print("📊 Feature Importance (Add to your PPT):")
for feature, importance in zip(X.columns, model.feature_importances_):
    print(f"- {feature}: {importance:.2%}")

joblib.dump(model, '../saved_models/landslide_model.pkl')
print("\n✅ Saved 'landslide_model.pkl'. Move this file to your backend folder or update the path in main.py!")

Booting up Data Pipeline...
Training XGBoost Classifier...
✅ Model Accuracy: 96.00%

📊 Feature Importance (Add to your PPT):
- rainfall_72h_mm: 32.77%
- slope_angle_deg: 53.64%
- elevation_m: 2.45%
- soil_moisture_index: 11.14%

✅ Saved 'landslide_model.pkl'. Move this file to your backend folder or update the path in main.py!
